Dataset: CMU-MOSEI

El dataset CMU-MOSEI es uno de los conjuntos de datos más utilizados en investigación para el análisis automático de emociones y sentimiento en vídeos.

Este dataset está compuesto por más de 23.500 clips de vídeo procedentes de más de 1000 personas diferentes, lo que garantiza una gran diversidad de expresiones, estilos de comunicación y contextos. Cada uno de estos vídeos ha sido anotado manualmente para reflejar tanto el sentimiento como diferentes emociones.

Una de las características más importantes de CMU-MOSEI es que es un dataset multimodal, es decir, incluye diferentes tipos de información para cada vídeo:

Audio: características extraídas de la voz (entonación, energía, etc.)
Texto: palabras, fonemas y representaciones semánticas
Visual: características faciales extraídas automáticamente
Etiquetas: anotaciones de sentimiento y emociones

En este notebook, sin embargo, no utilizaremos todas las modalidades. Nos centraremos únicamente en la parte visual, concretamente en el archivo:

CMU_MOSEI_VisualFacet42.csd
junto con sus etiquetas en CMU_MOSEI_Labels.csd

El archivo VisualFacet42 contiene, para cada vídeo, una secuencia temporal de características faciales. Es decir, en lugar de tener una sola fila por vídeo, tenemos información por frame. Por ejemplo, un vídeo puede representarse como una matriz de tamaño:

(número de frames, 35 características)

Esto significa que estamos capturando cómo evolucionan las expresiones faciales a lo largo del tiempo.

Por otro lado, el archivo de etiquetas contiene, para cada vídeo, un vector de 7 valores que representan:

sentimiento global
happy
sad
anger
fear
disgust
surprise

Es importante entender que estas etiquetas no son clases discretas, sino valores continuos que indican la intensidad de cada emoción.

En este notebook vamos a simplificar el problema transformándolo en una tarea de clasificación binaria. En concreto, nos interesa detectar si hay presencia de la emoción “happy” en el vídeo. Para ello, convertimos la etiqueta en:

1 si la intensidad de happy es mayor que 0
0 en caso contrario

Finalmente, el motivo por el que utilizamos únicamente la modalidad visual es porque queremos simular un escenario realista de despliegue en dispositivos con recursos limitados (Edge). Trabajar solo con información facial reduce la complejidad del sistema y facilita su implementación en entornos con restricciones de memoria y computación.

En este notebook cada vídeo se representará mediante características visuales agregadas, y el objetivo será entrenar modelos capaces de predecir si ese vídeo contiene o no la emoción “happy”.

Antes de empezar a trabajar con el modelo, es importante comprobar que nuestro entorno está correctamente configurado. El siguiente bloque de código nos permite verificar tres aspectos: la versión de Python, la versión de TensorFlow y la disponibilidad de GPU.

En primer lugar, se imprime la versión de Python que estamos utilizando. Esto es importante porque algunas librerías, como TensorFlow, requieren versiones específicas para funcionar correctamente.

A continuación, se muestra la versión de TensorFlow instalada. Saber qué versión estamos usando es fundamental para garantizar la compatibilidad del código y poder reproducir los resultados en otros entornos.

Por último, se comprueba si hay una GPU disponible en el sistema. TensorFlow permite utilizar GPU para acelerar el entrenamiento de modelos, lo que puede reducir significativamente los tiempos de ejecución. Si aparece una GPU en la salida, significa que el sistema está preparado para aprovechar aceleración hardware; en caso contrario, el entrenamiento se realizará en CPU.

Este tipo de comprobación es una buena práctica al inicio de cualquier proyecto de aprendizaje automático, ya que permite detectar problemas de configuración antes de ejecutar experimentos más costosos.

In [1]:
import sys
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
TensorFlow: 2.21.0
GPU disponible: []


Definición de rutas a los datos

Antes de poder trabajar con el dataset, necesitamos indicar al programa dónde se encuentran los archivos en nuestro sistema. Este bloque de código se encarga precisamente de eso: definir las rutas a los datos y comprobar que existen correctamente.

En primer lugar, utilizamos la librería pathlib, que es una forma moderna y segura de manejar rutas en Python. En lugar de trabajar con strings (cadenas de texto), Path nos permite construir rutas de forma más robusta y legible.

A continuación, definimos una variable llamada BASE_DIR, que representa la carpeta principal donde se encuentra el proyecto. A partir de esta carpeta base, construimos las rutas completas a los archivos que vamos a utilizar:

El archivo visual (CMU_MOSEI_VisualFacet42.csd)
El archivo de etiquetas (CMU_MOSEI_Labels.csd)

Esto se hace concatenando carpetas de forma estructurada:

BASE_DIR / "archive" / "CMU-MOSEI" / ...

Este enfoque tiene una ventaja importante: evita errores típicos al escribir rutas manualmente y hace el código más portable y fácil de modificar.

Finalmente, se realizan varias comprobaciones con .exists() para verificar que los archivos realmente están en la ubicación indicada. Esto es fundamental, ya que muchos errores en proyectos de datos vienen simplemente de rutas mal definidas o archivos que no están donde esperamos.

Además, se imprimen las rutas completas para que el usuario pueda confirmar visualmente que todo está correcto.

Para que este bloque funcione correctamente, es importante que el proyecto tenga una estructura de carpetas organizada. En este caso, se asume una estructura como la siguiente:

TFG_Emociones/
│
├── archive/
│   └── CMU-MOSEI/
│       ├── visuals/
│       │   └── CMU_MOSEI_VisualFacet42.csd
│       └── labels/
│           └── CMU_MOSEI_Labels.csd
│
├── notebooks/
├── models/
├── results/

Esto significa que los archivos del dataset deben colocarse exactamente en esas carpetas para que las rutas construidas en el código sean válidas.

Si estás ejecutando este notebook en tu propio ordenador, es muy probable que tengas que modificar la variable BASE_DIR. Esta variable debe apuntar a la carpeta raíz de tu proyecto, es decir, a la carpeta que contiene archive, notebooks, models, etc.

Por ejemplo, en este código aparece una ruta específica de un equipo concreto:

BASE_DIR = Path(r"C:\Users\Alberto Tena\OneDrive - udl.cat\Documentos\PROYECTOS_PYTHON\TFG_Emociones")

En tu caso, deberás sustituirla por la ruta donde hayas guardado el proyecto. Por ejemplo:

BASE_DIR = Path(r"C:\Users\TuUsuario\TFG_Emociones")

o en sistemas Linux/Mac:

BASE_DIR = Path("/home/usuario/TFG_Emociones")

Si al ejecutar el código ves que alguna de las comprobaciones devuelve False, significa que:

la ruta no es correcta, o
los archivos no están en la carpeta esperada

y debes revisar la estructura antes de continuar.

Este paso es fundamental, ya que todo el pipeline depende de poder acceder correctamente a los datos desde el inicio.

In [ ]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\pelli\Documents\TFG_Emociones")

VISUAL_PATH = BASE_DIR / "archive" / "CMU-MOSEI" / "visuals" / "CMU_MOSEI_VisualFacet42.csd"
LABELS_PATH = BASE_DIR / "archive" / "CMU-MOSEI" / "labels" / "CMU_MOSEI_Labels.csd"

print("Visual existe:", VISUAL_PATH.exists())
print("Labels existe:", LABELS_PATH.exists())
print("Visual:", VISUAL_PATH)
print("Labels:", LABELS_PATH)

Visual existe: True
Labels existe: True
Visual: C:\Users\pelli\Documents\TFG_Emociones\archive\CMU-MOSEI\visuals\CMU_MOSEI_VisualFacet42.csd
Labels: C:\Users\pelli\Documents\TFG_Emociones\archive\CMU-MOSEI\labels\CMU_MOSEI_Labels.csd


Inspección inicial de los archivos HDF5

Una vez que hemos definido correctamente las rutas a los datos, el siguiente paso es comprobar que los archivos pueden abrirse y entender su estructura interna.

Los archivos .csd del dataset CMU-MOSEI no son archivos de texto convencionales, sino que están basados en el formato HDF5. Este formato permite almacenar grandes cantidades de datos organizados de forma jerárquica, similar a un sistema de carpetas dentro de un archivo.

Para explorar estos archivos utilizamos la librería h5py, que nos permite abrirlos y navegar por su contenido.

En este bloque definimos una función llamada inspect_h5, cuyo objetivo es:

Intentar abrir el archivo
Verificar que es un archivo válido
Mostrar las claves principales (nivel superior de la jerarquía)

Cuando ejecutamos:

inspect_h5(VISUAL_PATH)
inspect_h5(LABELS_PATH)

el programa accede a cada archivo y muestra su estructura más básica.

Por ejemplo, podemos encontrar algo como:

Archivo HDF5 válido
Keys principales:
 - FACET 4.2

o en el caso de las etiquetas:

Keys principales:
 - All Labels

Estas “keys” son equivalentes a carpetas principales dentro del archivo. A partir de ellas, podremos seguir navegando hacia niveles más profundos donde se encuentran los datos reales.

Este paso es importante porque nos permite:

confirmar que los archivos no están corruptos
entender cómo están organizados los datos
preparar el siguiente paso, donde exploraremos el contenido en detalle

Si el archivo no se puede abrir, el bloque captura el error y lo muestra por pantalla, lo que facilita detectar problemas como rutas incorrectas o archivos dañados.

In [5]:
import h5py

def inspect_h5(path):
    try:
        with h5py.File(path, "r") as f:
            print("Archivo HDF5 válido")
            print("Keys principales:")
            for key in f.keys():
                print(" -", key)
    except Exception as e:
        print("No se ha podido abrir con h5py")
        print(type(e).__name__, ":", e)

inspect_h5(VISUAL_PATH)
inspect_h5(LABELS_PATH)

Archivo HDF5 válido
Keys principales:
 - FACET 4.2
Archivo HDF5 válido
Keys principales:
 - All Labels


Exploración detallada de la estructura interna

En el bloque anterior hemos visto las claves principales de los archivos. Ahora vamos un paso más allá: queremos explorar toda la estructura interna del archivo HDF5.

Para ello, definimos la función inspect_group, que recorre recursivamente el contenido del archivo utilizando el método visititems. Este método permite acceder a todos los elementos del archivo, independientemente de su nivel de profundidad.

Dentro de esta función, se define otra función auxiliar (print_structure) que se ejecuta para cada elemento encontrado. Esta función imprime:

el nombre del elemento (ruta dentro del archivo)
su tipo (grupo o dataset)
su forma (shape), en caso de que sea un dataset

Al ejecutar:

inspect_group(VISUAL_PATH)
inspect_group(LABELS_PATH)

obtenemos una vista completa de cómo están organizados los datos.

Este paso es importante para entender la estructura jerárquica del dataset. Por ejemplo, en el caso del archivo visual, veremos algo como:

FACET 4.2/data/<video_id>/features
FACET 4.2/data/<video_id>/intervals

Esto nos indica que:

Los datos están agrupados por vídeo (video_id)
Cada vídeo contiene:
features: las características faciales por frame
intervals: información temporal asociada a cada frame

En el caso de las etiquetas, la estructura será similar, pero con menos información temporal, ya que normalmente hay una única anotación por vídeo.

Este tipo de exploración es fundamental porque:

Nos permite entender cómo acceder a los datos correctamente
Evita suposiciones erróneas sobre la estructura
Facilita el diseño del pipeline de carga y procesamiento

Aunque la salida puede ser bastante larga (ya que hay miles de vídeos), lo importante no es leerla completa, sino identificar el patrón general de organización.

Este bloque nos da una visión completa de cómo están estructurados los datos dentro del archivo, lo cual es imprescindible antes de empezar a trabajar con ellos.

In [6]:
import h5py

def inspect_group(path):
    with h5py.File(path, "r") as f:
        def print_structure(name, obj):
            print(name, type(obj), getattr(obj, "shape", ""))

        f.visititems(print_structure)

print("=== VISUAL FACET 42 ===")
inspect_group(VISUAL_PATH)

print("\n=== LABELS ===")
inspect_group(LABELS_PATH)

=== VISUAL FACET 42 ===
FACET 4.2 <class 'h5py._hl.group.Group'> 
FACET 4.2/data <class 'h5py._hl.group.Group'> 
FACET 4.2/data/--qXJuDtHPw <class 'h5py._hl.group.Group'> 
FACET 4.2/data/--qXJuDtHPw/features <class 'h5py._hl.dataset.Dataset'> (1715, 35)
FACET 4.2/data/--qXJuDtHPw/intervals <class 'h5py._hl.dataset.Dataset'> (1715, 2)
FACET 4.2/data/-3g5yACwYnA <class 'h5py._hl.group.Group'> 
FACET 4.2/data/-3g5yACwYnA/features <class 'h5py._hl.dataset.Dataset'> (4340, 35)
FACET 4.2/data/-3g5yACwYnA/intervals <class 'h5py._hl.dataset.Dataset'> (4340, 2)
FACET 4.2/data/-3nNcZdcdvU <class 'h5py._hl.group.Group'> 
FACET 4.2/data/-3nNcZdcdvU/features <class 'h5py._hl.dataset.Dataset'> (1328, 35)
FACET 4.2/data/-3nNcZdcdvU/intervals <class 'h5py._hl.dataset.Dataset'> (1328, 2)
FACET 4.2/data/-571d8cVauQ <class 'h5py._hl.group.Group'> 
FACET 4.2/data/-571d8cVauQ/features <class 'h5py._hl.dataset.Dataset'> (2677, 35)
FACET 4.2/data/-571d8cVauQ/intervals <class 'h5py._hl.dataset.Dataset'> (2677

Cómo interpretar el output

Lo que estás viendo en pantalla es, literalmente, el mapa interno del archivo. Es decir, la estructura jerárquica completa donde se almacenan los datos.

La estructura sigue siempre este patrón:

[Raíz] / [Subcarpeta] / [ID del vídeo] / [Tipo de dato]

Por ejemplo:

All Labels/data/kqXpf26EL-s/features

Se interpreta así:

All Labels → raíz del archivo (nombre del dataset)
data → carpeta que contiene todos los ejemplos
kqXpf26EL-s → identificador único del vídeo (ID de YouTube)
features → datos asociados a ese vídeo

Este ID es fundamental, porque es el que utilizaremos para emparejar los datos visuales con sus etiquetas correspondientes.

Qué contiene cada vídeo

Dentro de cada vídeo encontramos dos elementos:

1. intervals

Tiene forma:

(n, 2)

Indica los intervalos temporales del vídeo:

cada fila es un segmento
contiene: [inicio, fin] en segundos

Ejemplo:

(10, 2)

→ el vídeo está dividido en 10 segmentos

2. features

Tiene forma:

(n, 7)

Aquí están las etiquetas reales.

El número 7 corresponde a:

Sentiment (valor continuo de -3 a +3)
Happy
Sad
Anger
Fear
Disgust
Surprise

Cada fila corresponde a un segmento del vídeo.

Qué significa esto para el proyecto

No tenemos una única etiqueta por vídeo, sino potencialmente varias (una por segmento).

Sin embargo, en nuestro caso hemos observado que muchos vídeos tienen:

(1, 7)

→ una sola etiqueta por vídeo

Esto simplifica mucho el problema.

Cómo lo vamos a usar

Para construir el modelo, vamos a:

recorrer todos los video_id
acceder a features
seleccionar la dimensión de interés (en nuestro caso, happy)

Y convertirlo en un problema binario:

happy > 0  → 1 (happy)
happy ≤ 0 → 0 (not happy)

En este punto del workflow damos un paso importante: pasamos de explorar todo el dataset de forma masiva a hacerlo de manera controlada y estratégica. Cuando trabajas con archivos HDF5 grandes como los de CMU-MOSEI, imprimir toda la estructura no es práctico. Por eso, este bloque introduce una forma más inteligente de inspeccionar los datos.

La función inspect_limited está diseñada para darte una visión clara del dataset sin saturarte de información. Primero abre el archivo y muestra las claves principales, lo que te permite identificar rápidamente los grandes bloques de datos (por ejemplo, etiquetas o características visuales). A partir de ahí, accede al grupo principal y calcula cuántos elementos contiene, que en este caso corresponden a los distintos vídeos del dataset.

En lugar de listar todos los vídeos, algo que podría generar miles de líneas, el código muestra solo una muestra de ellos (los primeros max_items). Esto es suficiente para entender cómo están organizados sin perder legibilidad.

Después, el código da un paso más interesante: selecciona un único vídeo (el primero) y lo analiza en detalle. Aquí es donde realmente entiendes cómo están organizados los datos a nivel individual. Verás que cada vídeo contiene dos elementos fundamentales:

features: donde están los datos (por ejemplo, emociones o características visuales)
intervals: que indican los segmentos temporales del vídeo

Además, se muestra información crítica como la forma (shape) y el tipo de dato (dtype), lo que te permite verificar que todo tiene sentido antes de procesarlo.

Este bloque cumple una función muy concreta dentro del flujo de trabajo: validar que los datos están bien estructurados y son accesibles antes de empezar a construir el dataset para el modelo. Es una práctica habitual, porque muchos errores posteriores (modelos que no entrenan, resultados inconsistentes, etc.) suelen venir de problemas en los datos que no se detectaron al principio.

En tu TFG, este paso tiene también un valor conceptual importante. No solo estás cargando datos, sino que estás demostrando que entiendes:

cómo están organizados los datos multimodales
cómo acceder a ellos de forma eficiente
cómo verificar su integridad antes de usarlos

Este bloque marca la transición entre “mirar datos” y empezar a trabajar con ellos de forma consciente y controlada, que es exactamente lo que necesitas antes de construir tu pipeline de entrenamiento.

In [7]:
import h5py

def inspect_limited(path, max_items=30):
    with h5py.File(path, "r") as f:
        print("Keys principales:", list(f.keys()))

        for main_key in f.keys():
            print("\nGrupo principal:", main_key)
            group = f[main_key]
            print("Tipo:", type(group))

            if isinstance(group, h5py.Group):
                keys = list(group.keys())
                print("Número de elementos:", len(keys))
                print("Primeras keys:", keys[:max_items])

                first_key = keys[0]
                first_item = group[first_key]
                print("\nPrimer elemento:", first_key)
                print("Tipo:", type(first_item))

                if isinstance(first_item, h5py.Group):
                    print("Subkeys:", list(first_item.keys()))
                    for subkey in first_item.keys():
                        obj = first_item[subkey]
                        print(subkey, type(obj), getattr(obj, "shape", None), getattr(obj, "dtype", None))
                else:
                    print("Shape:", getattr(first_item, "shape", None))
                    print("Dtype:", getattr(first_item, "dtype", None))

inspect_limited(VISUAL_PATH)
inspect_limited(LABELS_PATH)

Keys principales: ['FACET 4.2']

Grupo principal: FACET 4.2
Tipo: <class 'h5py._hl.group.Group'>
Número de elementos: 2
Primeras keys: ['data', 'metadata']

Primer elemento: data
Tipo: <class 'h5py._hl.group.Group'>
Subkeys: ['--qXJuDtHPw', '-3g5yACwYnA', '-3nNcZdcdvU', '-571d8cVauQ', '-6rXp3zJ3kc', '-9YyBTjo1zo', '-9y-fZ3swSY', '-AUZQgSxyPQ', '-Alixo7euuU', '-Eqdz5y4pEY', '-HeZS2-Prhc', '-HvKLjmsO5U', '-HwX2H8Z4hY', '-IUUR2yyNbw', '-I_e4mIh0yE', '-IqSFQePnpU', '-KCahx2qBOI', '-LnuDPiuuZw', '-MeTTeMJBNc', '-NFrJFQijFE', '-RfYyzHpjk4', '-RpZEe4w4fY', '-SYSVSQnTnA', '-THoVjtIkeU', '-UUCSKoHeMA', '-UacrmKiTn4', '-UuX1xuaiiE', '-VmheDA92mM', '-WXXTNIJcVM', '-ZgjBOA1Yhw', '-a55Q6RWvTA', '-aNfi7CP8vM', '-aqamKhZ1Ec', '-bl5PfNIYrk', '-cEhr0cQcDM', '-cmk6cfUeMs', '-dZ1TCboxcQ', '-dxfTGcXJoc', '-egA8-b7-3M', '-hPfPhUIzfA', '-hnBHBN8p5A', '-iRBcNs9oI8', '-l_53IwQoj0', '-lqc32Zpr7M', '-lzEya4AM_4', '-m9KtvCk_L8', '-mJ2ud6oKI8', '-mqbVkbCndg', '-o_667Ny7RM', '-qDkUB0GgYY', '-qMhGeJkp5o', '-ri04Z7v

En este bloque damos un paso más concreto: dejamos de analizar la estructura general del archivo y empezamos a trabajar con datos reales de un vídeo específico. A partir de aquí ya no estamos viendo el “mapa”, sino un ejemplo real que luego usaremos para construir el dataset de entrenamiento.

Primero abrimos el archivo HDF5 de las características visuales (VisualFacet42) y accedemos directamente a la ruta donde están los datos:

f["FACET 4.2"]["data"]

Aquí es importante entender que:

"FACET 4.2" es el nombre del bloque de características visuales
"data" contiene todos los vídeos del dataset

Cada elemento dentro de data corresponde a un vídeo, identificado por su video_id (que coincide con el ID de YouTube).

A continuación, el código selecciona el primer vídeo disponible:

first_video_id = list(data_group.keys())[0]

Esto no tiene nada especial en sí (podría ser cualquier otro), simplemente nos sirve como ejemplo para inspeccionar cómo están organizados los datos a nivel individual.

Una vez seleccionado, accedemos a su contenido:

video_group = data_group[first_video_id]

Y mostramos sus subclaves:

Subkeys: ['features', 'intervals']

Aquí aparece de nuevo una estructura que ya habíamos visto en las etiquetas, pero ahora aplicada a las características visuales. Esto es muy importante porque confirma que ambas fuentes de datos (visual y labels) están alineadas estructuralmente.

Finalmente, recorremos cada elemento (features e intervals) e imprimimos la siguiente información:

el tipo de objeto
la forma (shape)
el tipo de dato (dtype)

Esto te permite interpretar correctamente qué estás viendo:

features
Contiene las características visuales extraídas frame a frame (por ejemplo, información sobre expresiones faciales).

Su forma suele ser (n, 35) en este dataset, donde:
n es el número de frames o segmentos
35 es el número de características por frame

intervals
Indica los intervalos temporales asociados a esos frames:
cada fila es [inicio, fin] en segundos
su forma es (n, 2)

Este punto es especialmente relevante para tu TFG, porque aquí entiendes algo fundamental: los datos visuales son secuencias temporales, no vectores fijos. Es decir, cada vídeo tiene una longitud distinta.

Esto tiene implicaciones directas en el modelo:

tendrás que decidir cómo convertir estas secuencias en un formato uniforme (por ejemplo, agregación, padding, pooling, etc.)
o bien diseñar un modelo que trabaje con secuencias (RNN, Transformers, etc.).

In [8]:
import h5py

with h5py.File(VISUAL_PATH, "r") as f:
    data_group = f["FACET 4.2"]["data"]

    first_video_id = list(data_group.keys())[0]
    print("Primer video_id:", first_video_id)

    video_group = data_group[first_video_id]
    print("Subkeys:", list(video_group.keys()))

    for key in video_group.keys():
        obj = video_group[key]
        print(
            key,
            "type:", type(obj),
            "shape:", getattr(obj, "shape", None),
            "dtype:", getattr(obj, "dtype", None)
        )

Primer video_id: --qXJuDtHPw
Subkeys: ['features', 'intervals']
features type: <class 'h5py._hl.dataset.Dataset'> shape: (1715, 35) dtype: float32
intervals type: <class 'h5py._hl.dataset.Dataset'> shape: (1715, 2) dtype: float64


En este bloque repetimos exactamente la misma lógica que en el caso visual, pero ahora aplicada a las etiquetas. Esto no es casual: forma parte de una idea clave en este dataset, y es que los datos visuales y las etiquetas están estructurados de forma paralela, lo que facilita su emparejamiento posterior.

Primero abrimos el archivo de etiquetas (CMU_MOSEI_Labels.csd) y accedemos a la ruta principal:

f["All Labels"]["data"]

Aquí ocurre algo importante:

"All Labels" es el bloque donde se almacenan las anotaciones
"data" contiene todas las muestras, organizadas por video_id

Al igual que antes, cada video_id corresponde a un vídeo concreto, y coincide con los IDs que vimos en el archivo visual. Esta correspondencia es fundamental, porque será la clave para construir nuestro dataset final.

Después seleccionamos un ejemplo:

first_label_id = list(labels_group.keys())[0]

Y accedemos a su contenido:

label_group = labels_group[first_label_id]

De nuevo aparecen las mismas subclaves:

features
intervals

Esto confirma que ambas modalidades (visual y etiquetas) siguen la misma organización temporal.

Cuando imprimimos la información de cada elemento, lo más relevante es interpretar correctamente features.

Aquí es donde está la diferencia clave respecto al bloque visual:

features (forma: n, 7)
Contiene las etiquetas emocionales y de sentimiento
Cada fila corresponde a un segmento del vídeo

Las 7 columnas representan:

Sentiment (valor continuo, típicamente entre -3 y +3)
Happy
Sad
Anger
Fear
Disgust
Surprise
intervals (forma: n, 2)
Igual que antes, indica los intervalos temporales de cada segmento

Idea clave para el workflow

Aquí es donde todo empieza a encajar:

El archivo visual tiene features por frame/segmento
El archivo de etiquetas tiene emociones por segmento
Ambos usan el mismo video_id y la misma estructura temporal

Esto significa que puedes alinear directamente:

Visual features  ⟷  Label features

Cómo lo vamos a usar

Para tu objetivo (clasificar happy vs not happy), lo que haremos más adelante será:

recorrer todos los video_id
acceder a features en labels
seleccionar la columna correspondiente a happy
convertirla a binario:
valor > 0 → clase 1 (happy)
valor ≤ 0 → clase 0 (not happy)

Este paso no es solo técnico, es conceptual. Aquí entiendes:

cómo están representadas las emociones en el dataset
que las etiquetas no son una única clase, sino múltiples dimensiones continuas
que los datos están organizados temporalmente
y que existe una correspondencia directa entre modalidades

Este bloque es el que te permite pasar de “tener datos” a saber exactamente qué significa cada número y cómo usarlo en tu modelo.

In [9]:
with h5py.File(LABELS_PATH, "r") as f:
    labels_group = f["All Labels"]["data"]

    first_label_id = list(labels_group.keys())[0]
    print("Primer label_id:", first_label_id)

    label_group = labels_group[first_label_id]
    print("Subkeys:", list(label_group.keys()))

    for key in label_group.keys():
        obj = label_group[key]
        print(
            key,
            "type:", type(obj),
            "shape:", getattr(obj, "shape", None),
            "dtype:", getattr(obj, "dtype", None)
        )

Primer label_id: --qXJuDtHPw
Subkeys: ['features', 'intervals']
features type: <class 'h5py._hl.dataset.Dataset'> shape: (1, 7) dtype: float32
intervals type: <class 'h5py._hl.dataset.Dataset'> shape: (1, 2) dtype: float64


En el caso del archivo visual, la salida indica:
features = (1715, 35)

Esto significa que este vídeo contiene 1715 instantes temporales o frames procesados, y que para cada uno de ellos se han extraído 35 características visuales. Estas características describen información facial relevante, como patrones asociados a expresiones o movimientos faciales.

Por tanto, el vídeo no está representado como imágenes, sino como una secuencia de vectores numéricos:
vídeo → 1715 frames → 35 características por frame

En el caso del archivo de etiquetas, la salida indica:
features = (1, 7)

Esto significa que para este vídeo tenemos un vector de etiquetas con 7 valores. Estos valores corresponden a las dimensiones emocionales/anotaciones del dataset:

sentiment, happy, sad, anger, fear, disgust, surprise

En resumen, para este vídeo concreto tenemos:

Entrada visual: secuencia de características faciales
Etiqueta: vector emocional asociado al vídeo

Esto nos permite construir un modelo supervisado: las características visuales serán la entrada (X) y la emoción seleccionada, en nuestro caso happy, será la etiqueta objetivo (y).

En este bloque construimos el dataset final que utilizará el modelo, y es probablemente el paso más importante de todo el pipeline. Aquí es donde transformamos los datos complejos del dataset (secuencias temporales multimodales) en una representación estructurada y lista para aplicar técnicas de machine learning.

La función build_visual_dataset recorre simultáneamente los datos visuales y las etiquetas, asegurándose de que ambas fuentes estén correctamente alineadas. Para ello, primero obtiene los video_id que existen en ambos archivos. Esto garantiza que cada muestra tenga tanto características de entrada como su etiqueta correspondiente.

A partir de ahí, el proceso se realiza vídeo a vídeo. Para cada uno, se cargan:

las características visuales (features), que tienen forma (n, 35)
las etiquetas (features en labels), que tienen forma (1, 7)

En este punto aparece uno de los principales retos: cada vídeo tiene un número distinto de frames (n). Sin embargo, los modelos de scikit-learn requieren vectores de tamaño fijo. Para resolver esto, aplicamos una estrategia sencilla pero efectiva: agregación temporal.

En lugar de trabajar con toda la secuencia, resumimos la información del vídeo calculando:

la media de cada característica
la desviación estándar de cada característica

Esto permite capturar tanto el valor promedio como la variabilidad a lo largo del tiempo. Después, ambos vectores se concatenan en uno solo, dando como resultado un vector final de tamaño 70:

35 (media) + 35 (desviación estándar) = 70 características

De esta forma, cada vídeo queda representado por un único vector fijo, independientemente de su duración.

Antes de esta agregación, además, se realiza una limpieza básica de los datos usando np.nan_to_num, que elimina posibles valores inválidos como NaN o infinitos. Este paso es fundamental para evitar problemas durante el entrenamiento del modelo.

En paralelo, se almacenan las etiquetas originales (y_raw). Es importante notar que aquí no estamos transformando todavía las etiquetas, simplemente guardamos el vector completo de 7 dimensiones. Más adelante seleccionaremos la dimensión que nos interesa (por ejemplo, happy) y la convertiremos en un problema binario.

Finalmente, todos los datos se convierten en arrays de NumPy, obteniendo:

X: matriz de características con forma (N, 70)
y_raw: matriz de etiquetas con forma (N, 7)
video_ids: lista de identificadores

Donde N es el número de vídeos válidos procesados.

Cuando imprimimos:

X shape: (N, 70)
y_raw shape: (N, 7)

esto confirma que:

cada fila de X representa un vídeo
cada fila de y_raw contiene sus 7 etiquetas emocionales

Este bloque marca un punto de inflexión en el workflow. Hemos pasado de trabajar con datos crudos y estructuras complejas a tener un dataset limpio, consistente y preparado para entrenar modelos. A partir de aquí, los siguientes pasos serán:

seleccionar la emoción objetivo (happy)
convertirla en un problema binario (happy vs not happy)
aplicar validación cruzada, balanceo de clases y selección de características
entrenar y comparar modelos

En definitiva, este bloque convierte el dataset en algo utilizable para aprendizaje automático, y define la base sobre la que construiremos todo el sistema de clasificación.

In [12]:
import h5py
import numpy as np

def build_visual_dataset(visual_path, labels_path):
    X = []
    y_raw = []
    video_ids = []

    with h5py.File(visual_path, "r") as vf, h5py.File(labels_path, "r") as lf:
        visual_data = vf["FACET 4.2"]["data"]
        label_data = lf["All Labels"]["data"]

        common_ids = sorted(set(visual_data.keys()) & set(label_data.keys()))

        for vid in common_ids:
            visual_features = visual_data[vid]["features"][:]
            label_features = label_data[vid]["features"][:]

            if visual_features.size == 0 or label_features.size == 0:
                continue

            # Eliminar posibles NaN o infinitos
            visual_features = np.nan_to_num(visual_features)

            # Agregación temporal sencilla
            mean_features = np.mean(visual_features, axis=0)
            std_features = np.std(visual_features, axis=0)

            final_features = np.concatenate([mean_features, std_features])

            X.append(final_features)
            y_raw.append(label_features[0])
            video_ids.append(vid)

    return np.array(X), np.array(y_raw), video_ids


X, y_raw, video_ids = build_visual_dataset(VISUAL_PATH, LABELS_PATH)

print("X shape:", X.shape)
print("y_raw shape:", y_raw.shape)
print("Número de vídeos:", len(video_ids))
print("Primera etiqueta:", y_raw[0])

X shape: (3293, 70)
y_raw shape: (3293, 7)
Número de vídeos: 3293
Primera etiqueta: [1.        0.6666667 0.        0.        0.        0.        0.       ]


En este bloque realizamos una comprobación: visualizar directamente las etiquetas que hemos construido.

El código: print("Primeras 10 etiquetas:")print(y_raw[:10])
simplemente muestra las primeras 10 filas de y_raw, es decir, las etiquetas asociadas a los primeros vídeos del dataset.

🧠 ¿Qué estamos viendo exactamente?

Cada fila que aparece tiene 7 valores, por ejemplo:

[ 1.          0.6666667   0.          0.          0.          0.          0.        ]

Esto corresponde a:

[sentiment, happy, sad, anger, fear, disgust, surprise]

Es importante entender que:

no son etiquetas binarias (0 o 1)

son valores continuos, que representan intensidad


Al mirar estas etiquetas aparecen cosas interesantes:

valores positivos → indican presencia de esa emoción

valores 0 → ausencia

valores negativos → posible ausencia o polaridad contraria (especialmente en sentiment)

valores como 0.6666, 1.3333, etc. → diferentes niveles de intensidad

Esto confirma que el problema original es de regresión multietiqueta, no de clasificación directa.

Conexión con nuestro objetivo

Aquí es donde tomamos una decisión importante de modelado.

Nosotros no queremos predecir todas las emociones ni trabajar con valores continuos. Vamos a simplificar el problema a:

¿Está feliz o no está feliz?

Por tanto, en el siguiente paso haremos:

seleccionar la columna happy

convertirla a binaria:

> 0 → 1 (happy)

≤ 0 → 0 (not happy)


Este bloque sirve para:

verificar que las etiquetas se han cargado correctamente

entender su distribución y naturaleza

detectar posibles problemas (valores raros, negativos, etc.)

Y, sobre todo, para asegurarte de que sabes exactamente qué estás prediciendo antes de entrenar el modelo.

Aquí pasas de “tener etiquetas” a entender qué significan realmente y cómo las vas a transformar para tu problema de clasificación.

In [ ]:
print("Primeras 10 etiquetas:")
print(y_raw[:10])

En este bloque analizamos la distribución de las emociones en todo el dataset, algo fundamental antes de entrenar cualquier modelo.

El código recorre cada una de las 7 etiquetas emocionales y, para cada una, cuenta cuántos vídeos tienen un valor positivo. Es decir, estamos interpretando que una emoción está presente cuando su valor es mayor que 0.

Esto nos permite pasar de ver ejemplos individuales a tener una visión global del dataset.

Lo que estamos midiendo es, en esencia:

cuántas veces aparece cada emoción
si hay emociones mucho más frecuentes que otras

Este análisis es importante porque introduce el concepto de desbalanceo de clases. No todas las emociones aparecen con la misma frecuencia, y esto afecta directamente al rendimiento del modelo.

Por ejemplo, si una emoción aparece muy poco, el modelo puede tener dificultades para detectarla correctamente. En cambio, si aparece mucho, el modelo puede tender a predecirla con mayor facilidad, incluso cuando no debería.

En tu caso, este bloque tiene un objetivo muy concreto: validar si la emoción happy es adecuada para construir un clasificador binario (happy vs not happy). Si tiene suficientes ejemplos positivos y no está extremadamente desbalanceada, es una buena elección.

Aquí estás haciendo un análisis exploratorio básico pero esencial, que te permite:

entender cómo están distribuidas las etiquetas
anticipar posibles problemas en el entrenamiento
tomar decisiones informadas sobre el diseño del modelo

Este tipo de análisis es lo que diferencia un pipeline bien planteado de uno que simplemente “entrena sin mirar los datos”.

In [13]:
emotion_names = ["sentiment", "happy", "sad", "anger", "fear", "disgust", "surprise"]

for i, name in enumerate(emotion_names):
    scores = y_raw[:, i]
    positives = np.sum(scores > 0)
    print(name, "positivos:", positives, "de", len(scores))

sentiment positivos: 1887 de 3293
happy positivos: 2008 de 3293
sad positivos: 726 de 3293
anger positivos: 592 de 3293
fear positivos: 240 de 3293
disgust positivos: 283 de 3293
surprise positivos: 368 de 3293


Pipeline experimental y buenas prácticas

A partir de este punto, el objetivo ya no es solo cargar los datos, sino construir un pipeline experimental correcto y reproducible.

El flujo completo que vamos a seguir es:

VisualFacet42 features
↓
agregación temporal por vídeo
↓
etiqueta binaria: happy / not happy
↓
k-fold estratificado
↓
normalización dentro de cada fold
↓
selección de mejores características dentro de cada fold
↓
balanceo de clases
↓
entrenamiento: modelo normal vs modelo tiny
↓
evaluación comparativa

Este pipeline está diseñado para evitar uno de los errores más comunes en machine learning: el data leakage. Esto ocurre cuando información del conjunto de test se utiliza, directa o indirectamente, durante el entrenamiento. Para evitarlo, todas las transformaciones deben ajustarse únicamente con los datos de entrenamiento de cada fold.

En concreto:

StandardScaler se ajusta solo con X_train
SelectKBest se ajusta solo con X_train e y_train
class_weight se calcula solo con y_train
el conjunto de test de cada fold no se toca hasta la evaluación final

El objetivo experimental no es simplemente entrenar un modelo, sino comparar dos alternativas: un modelo de mayor tamaño y un modelo tiny. La idea es analizar si el modelo tiny puede mantener un rendimiento competitivo reduciendo la complejidad computacional.

Para ello evaluaremos métricas como:

accuracy
precision
recall
F1-score
AUC
latencia
número de parámetros

La métrica principal será el F1-score, ya que existe cierto desbalance entre las clases happy y not happy.

Este experimento busca responder a una pregunta concreta:

¿Puede un modelo tiny detectar la emoción happy con un rendimiento similar al modelo normal, pero con menor coste computacional y menor latencia?

En este bloque damos un paso importante en el pipeline: transformamos el problema original en uno de clasificación binaria.

Hasta ahora, las etiquetas (y_raw) contienen 7 dimensiones emocionales con valores continuos. Sin embargo, nuestro objetivo no es predecir todas las emociones, sino centrarnos en una sola: happy.

Para ello, seleccionamos la columna correspondiente:

happy_scores = y_raw[:, 1]

Esto extrae, para todos los vídeos, la intensidad de la emoción happy.

A continuación, convertimos estos valores continuos en una etiqueta binaria:

y = (happy_scores > 0).astype(int)

La lógica es simple:

valor > 0 → 1 (happy)
valor ≤ 0 → 0 (not happy)

De esta forma, transformamos un problema de regresión multietiqueta en un problema de clasificación binaria, mucho más manejable y alineado con nuestro objetivo experimental.

Después, se imprimen tres elementos para validar que todo es correcto:

X shape: confirma que las características no han cambiado
y shape: ahora tenemos un vector de etiquetas binarias
np.bincount(y): muestra cuántos ejemplos hay de cada clase

Este último punto es especialmente importante, porque nos permite ver la distribución de clases:

cuántos vídeos son not happy
cuántos son happy


Aquí estás definiendo exactamente qué va a aprender el modelo. Además:

simplificas el problema
haces posible usar clasificadores estándar
preparas el terreno para aplicar métricas como F1-score

También empiezas a ver si existe desbalanceo entre clases, lo cual influirá en decisiones posteriores como el uso de class_weight.

Este bloque convierte las etiquetas originales en el target final del modelo, alineado con el objetivo del experimento: detectar automáticamente si un vídeo expresa felicidad o no.

In [14]:
# ============================================================
# 1. Crear etiqueta binaria: happy vs not happy
# ============================================================

import numpy as np

happy_scores = y_raw[:, 1]   # columna 1 = happy
y = (happy_scores > 0).astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Distribución [not happy, happy]:", np.bincount(y))

X shape: (3293, 70)
y shape: (3293,)
Distribución [not happy, happy]: [1285 2008]


El output que se muestra aquí es una verificación directa de que el dataset se ha construido correctamente y está listo para entrenar modelos.

Primero:

X shape: (3293, 70)

Esto indica que tenemos 3293 muestras (vídeos) y que cada una está representada por un vector de 70 características. Estas 70 características provienen de la agregación temporal:

35 valores de media
35 valores de desviación estándar

Es decir, cada vídeo ha sido transformado en una representación fija y compacta.

Después:

y shape: (3293,)

Esto significa que tenemos una etiqueta por cada vídeo, lo cual es correcto en un problema supervisado. El vector y es ahora un array unidimensional con valores binarios (0 o 1).

Finalmente:

Distribución [not happy, happy]: [1285 2008]

Aquí vemos cómo se distribuyen las clases:

1285 vídeos → not happy (clase 0)
2008 vídeos → happy (clase 1)

Esto nos da varias conclusiones importantes:

el dataset no está perfectamente balanceado
hay más ejemplos de happy que de not happy
la proporción es aproximadamente 60% vs 40%

Implicaciones para el modelo

Este ligero desbalanceo justifica decisiones que ya hemos planteado en el pipeline:

usar métricas como F1-score en lugar de solo accuracy
aplicar class_weight durante el entrenamiento
utilizar k-fold estratificado para mantener la proporción de clases

Este output confirma tres cosas fundamentales:

los datos están correctamente estructurados
las etiquetas están alineadas con las muestras
existe un desbalance moderado que debemos tener en cuenta durante el entrenamiento

Con esto, el dataset ya está completamente preparado para pasar a la fase de validación cruzada y entrenamiento de modelos.

En este bloque preparamos todas las herramientas necesarias para el entrenamiento y evaluación del modelo. No hay procesamiento de datos todavía, pero sí estamos definiendo el conjunto de librerías que usaremos en el resto del pipeline.

Primero se importan utilidades generales:

time: nos permitirá medir tiempos de ejecución, especialmente útil para evaluar latencia
pandas: para organizar resultados en tablas y facilitar el análisis
tensorflow: lo utilizaremos para construir y entrenar los modelos (normal y tiny)

Después aparecen los componentes de scikit-learn, que implementan las buenas prácticas que hemos definido:

StratifiedKFold: Permite dividir los datos en varios folds manteniendo la proporción de clases (happy / not happy). Esto es fundamental cuando hay desbalanceo.

StandardScaler: Se utiliza para normalizar las características (media 0, desviación estándar 1). Esto ayuda a que muchos modelos aprendan mejor.

SelectKBest + f_classif: Permiten seleccionar las características más relevantes según su relación con la variable objetivo. Esto reduce ruido y puede mejorar el rendimiento.

compute_class_weight: Calcula pesos para cada clase en función de su frecuencia. Es una técnica común para compensar el desbalanceo.

Finalmente, se importan las métricas de evaluación:

accuracy: proporción de aciertos
precision: qué porcentaje de predicciones positivas son correctas
recall: qué porcentaje de positivos reales se detectan
f1_score: equilibrio entre precision y recall (nuestra métrica principal)
roc_auc_score: mide la capacidad de discriminación del modelo
classification_report: resumen completo de métricas por clase


Este bloque define el ecosistema experimental:

cómo validamos (K-Fold estratificado)
cómo preprocesamos (normalización y selección de características)
cómo manejamos el desbalanceo
cómo evaluamos el rendimiento

Todo lo que viene después en el pipeline se apoya directamente en estos componentes.

Aquí estás preparando las herramientas necesarias para construir un experimento riguroso, reproducible y alineado con buenas prácticas en machine learning.

In [15]:
# ============================================================
# 2. Imports
# ============================================================

import time
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

En este bloque definimos los dos modelos que vamos a comparar: un modelo más complejo (normal) y uno simplificado (tiny). Esta comparación es el núcleo de tu objetivo experimental.

Ambos modelos están implementados con redes neuronales densas (MLP), ya que nuestras entradas (X) son vectores numéricos de tamaño fijo.


Modelo normal

El modelo build_normal_model tiene mayor capacidad de aprendizaje. Su arquitectura es más profunda y con más neuronas:

varias capas densas: 256 → 128 → 64
capas de Dropout para evitar sobreajuste
función de activación ReLU en las capas ocultas
capa final con sigmoid para clasificación binaria

La idea de este modelo es:

capturar relaciones complejas entre las características
maximizar el rendimiento, aunque sea a costa de mayor coste computacional


Modelo tiny

El modelo build_tiny_model es mucho más ligero:

menos capas
muchas menos neuronas: 32 → 16
sin Dropout

Esto implica:

menor número de parámetros
menor tiempo de inferencia (menor latencia)
menor consumo de recursos

Pero también:

menor capacidad para modelar patrones complejos


Elementos comunes

Ambos modelos comparten decisiones importantes:

Entrada: input_dim (las 70 características)
Función de salida: sigmoid
produce un valor entre 0 y 1 (probabilidad de happy)
Función de pérdida: binary_crossentropy
adecuada para clasificación binaria
Optimizador: Adam con learning_rate = 1e-3
Métrica durante entrenamiento: accuracy

Qué estamos comparando realmente?

Este diseño permite evaluar un compromiso clásico en machine learning:

modelo normal → más precisión potencial
modelo tiny → más eficiente (menos latencia y coste)

La pregunta que queremos responder es:

¿Podemos reducir drásticamente la complejidad del modelo sin perder demasiado rendimiento en la detección de happy?


Idea importante para tu TFG

Este bloque define el experimento comparativo. No estás probando un solo modelo, sino evaluando dos enfoques con objetivos distintos:

rendimiento máximo
eficiencia computacional

Y esto encaja directamente con tu objetivo final: encontrar un modelo que sea viable para entornos edge, donde la latencia y los recursos son limitados.

Aquí estás estableciendo las dos “opciones” que luego vas a evaluar de forma rigurosa en el pipeline.

In [16]:
# ============================================================
# 3. Definir modelos normal y tiny
# ============================================================

from tensorflow.keras import layers, models

def build_normal_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def build_tiny_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

En este bloque definimos dos funciones auxiliares que son fundamentales para que el experimento sea correcto y completo: una para manejar el desbalanceo de clases y otra para evaluar los modelos de forma rigurosa.


Cálculo de pesos de clase

La función compute_class_weights se encarga de calcular automáticamente cuánto peso debe tener cada clase durante el entrenamiento.

La idea es sencilla: si una clase aparece menos veces, se le da más importancia para que el modelo no la ignore.

Internamente:

obtiene las clases presentes (0 y 1)
calcula pesos inversamente proporcionales a su frecuencia
devuelve un diccionario tipo:
{0: peso_not_happy, 1: peso_happy}

Esto se usará en el entrenamiento del modelo para compensar el desbalanceo que vimos anteriormente.


Evaluación del modelo

La función evaluate_model es la que mide el rendimiento real del modelo sobre el conjunto de test.

Primero realiza la predicción:

y_prob = model.predict(X_test)

Esto devuelve probabilidades (valores entre 0 y 1). Después se convierten en clases binarias usando un umbral de 0.5:

y_pred = (y_prob >= 0.5).astype(int)


Medición de latencia

Un aspecto muy importante en tu experimento es la eficiencia. Por eso se mide también la latencia:

latency_ms = ((end - start) / len(X_test)) * 1000

Esto calcula el tiempo medio que tarda el modelo en procesar una muestra (en milisegundos). Es importante para comparar el modelo normal frente al tiny.


Métricas calculadas

Se calculan varias métricas para evaluar el rendimiento:

accuracy: proporción de aciertos
precision: calidad de las predicciones positivas
recall: capacidad de detectar positivos reales
f1_score: equilibrio entre precision y recall (la más importante en tu caso)
latency_ms_per_sample: eficiencia del modelo

Además, se intenta calcular:

AUC: capacidad de separar clases usando probabilidades

Si no es posible calcular AUC (por ejemplo, si solo hay una clase en y_test), se asigna NaN para evitar errores.



Qué aporta este bloque al pipeline?

Aquí estás definiendo cómo:

manejar el desbalanceo de forma automática
evaluar los modelos de forma completa (rendimiento + eficiencia)

Esto es importante porque tu experimento no solo busca precisión, sino también viabilidad práctica.


Idea importante

Este bloque conecta directamente con tu objetivo final:

compute_class_weights → mejora el aprendizaje en datos desbalanceados
evaluate_model → permite comparar modelos de forma justa

Aquí defines cómo medir correctamente qué modelo es mejor, no solo en términos de rendimiento, sino también de coste computacional.

In [17]:
# ============================================================
# 4. Funciones auxiliares
# ============================================================

def compute_class_weights(y_train):
    classes = np.unique(y_train)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    return dict(zip(classes, weights))


def evaluate_model(model, X_test, y_test):
    start = time.time()
    y_prob = model.predict(X_test, verbose=0).flatten()
    end = time.time()

    y_pred = (y_prob >= 0.5).astype(int)

    latency_ms = ((end - start) / len(X_test)) * 1000

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "latency_ms_per_sample": latency_ms
    }

    try:
        metrics["auc"] = roc_auc_score(y_test, y_prob)
    except ValueError:
        metrics["auc"] = np.nan

    return metrics, y_pred, y_prob

Este bloque es el núcleo experimental del notebook. Aquí se entrena y evalúa todo el pipeline siguiendo buenas prácticas de machine learning: validación cruzada, normalización, selección de características, balanceo de clases y comparación entre el modelo normal y el modelo tiny.

Primero se definen los parámetros generales del experimento. Usamos N_SPLITS = 5, lo que significa que aplicaremos una validación cruzada de 5 folds. En cada iteración, una parte del dataset se usará como test y el resto como entrenamiento. Esto permite obtener una evaluación más robusta que una única división train/test.

También se define:

K_FEATURES = 50: seleccionamos las 50 mejores características de las 70 disponibles.
EPOCHS = 40: número máximo de épocas de entrenamiento.
BATCH_SIZE = 32: tamaño de lote.
RANDOM_STATE = 42: semilla para reproducibilidad.

A continuación, se crea el objeto StratifiedKFold. La palabra stratified es importante: significa que cada fold mantiene aproximadamente la misma proporción de clases happy y not happy. Esto es especialmente útil cuando existe cierto desbalanceo entre clases.

Después se inicializan varias variables:

results: almacenará las métricas de todos los experimentos.
best_tiny_model: guardará el mejor modelo tiny encontrado.
best_tiny_f1: almacenará el mejor F1-score obtenido por el modelo tiny.
best_fold: indicará en qué fold se obtuvo ese mejor resultado.

Dentro del bucle principal, el código recorre los 5 folds. En cada fold se separan los datos en entrenamiento y test:

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

A partir de aquí se aplican las transformaciones importantes, siempre ajustándolas solo con los datos de entrenamiento. Esto es clave para evitar data leakage.

Primero se normalizan las características con StandardScaler. El escalador aprende la media y desviación estándar de X_train, y después aplica esa misma transformación a X_test.

Después se aplica la selección de características con SelectKBest. Esta técnica selecciona las características más relevantes para distinguir entre happy y not happy. De nuevo, el selector se ajusta solo con X_train e y_train, y luego se aplica a X_test.

A continuación, se calculan los pesos de clase con compute_class_weights. Esto permite compensar el desbalanceo entre clases durante el entrenamiento, dando más peso a la clase menos representada.

Una vez preparado el fold, se entrenan los dos modelos:

normal_model
tiny_model

Ambos modelos se entrenan exactamente con los mismos datos y bajo las mismas condiciones, lo que hace que la comparación sea justa.

Durante el entrenamiento se usa EarlyStopping, que detiene el entrenamiento si la pérdida de validación deja de mejorar durante varias épocas. Esto ayuda a evitar sobreajuste y reduce tiempo de entrenamiento.

Después de entrenar cada modelo, se evalúa con la función evaluate_model, que devuelve métricas como accuracy, precision, recall, F1-score, AUC y latencia por muestra.

Cada resultado se guarda como una fila en el diccionario row, incluyendo:

número de fold
nombre del modelo
número de características usadas
número de parámetros
métricas de rendimiento
latencia

Finalmente, si el modelo evaluado es el tiny_model y consigue un F1-score superior al mejor obtenido hasta el momento, se guarda como el mejor modelo tiny.

Al final del bloque, todos los resultados se convierten en un DataFrame:

results_df = pd.DataFrame(results)

Este DataFrame es la base para comparar los modelos.

Este bloque responde a la pregunta central del experimento:

¿Qué modelo ofrece mejor equilibrio entre rendimiento predictivo y eficiencia computacional: el modelo normal o el modelo tiny?

Lo importante no es solo ver cuál tiene mayor accuracy, sino analizar conjuntamente:

F1-score
latencia
número de parámetros
estabilidad entre folds

Esta es la parte que permitirá tomar una decisión final justificada sobre qué modelo es más adecuado para un escenario Edge.

In [18]:
# ============================================================
# 5. K-Fold + selección de características + balanceo
# ============================================================

N_SPLITS = 5
K_FEATURES = min(50, X.shape[1])  # X tiene 70 features; probamos top 50
EPOCHS = 40
BATCH_SIZE = 32
RANDOM_STATE = 42

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

best_tiny_model = None
best_tiny_f1 = -1
best_fold = None

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n==============================")
    print(f"Fold {fold}/{N_SPLITS}")
    print(f"==============================")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # --------------------------------------------------------
    # Normalización SOLO con train para evitar data leakage
    # --------------------------------------------------------
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --------------------------------------------------------
    # Selección de características SOLO con train
    # --------------------------------------------------------
    selector = SelectKBest(score_func=f_classif, k=K_FEATURES)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)

    # --------------------------------------------------------
    # Balanceo de clases SOLO con train
    # --------------------------------------------------------
    class_weight = compute_class_weights(y_train)

    input_dim = X_train_selected.shape[1]

    model_builders = {
        "normal_model": build_normal_model,
        "tiny_model": build_tiny_model
    }

    for model_name, builder in model_builders.items():
        print(f"Entrenando {model_name}...")

        model = builder(input_dim)

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )

        model.fit(
            X_train_selected,
            y_train,
            validation_split=0.2,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=0
        )

        metrics, y_pred, y_prob = evaluate_model(
            model,
            X_test_selected,
            y_test
        )

        row = {
            "fold": fold,
            "model": model_name,
            "num_features": input_dim,
            "num_parameters": model.count_params(),
            **metrics
        }

        results.append(row)

        print(model_name, row)

        if model_name == "tiny_model" and metrics["f1_score"] > best_tiny_f1:
            best_tiny_f1 = metrics["f1_score"]
            best_tiny_model = model
            best_fold = fold

results_df = pd.DataFrame(results)
results_df


Fold 1/5
Entrenando normal_model...
normal_model {'fold': 1, 'model': 'normal_model', 'num_features': 50, 'num_parameters': 54273, 'accuracy': 0.6433990895295902, 'precision': 0.7537993920972644, 'recall': 0.6169154228855721, 'f1_score': 0.6785225718194254, 'latency_ms_per_sample': 0.12629961931289418, 'auc': 0.713330235979635}
Entrenando tiny_model...
tiny_model {'fold': 1, 'model': 'tiny_model', 'num_features': 50, 'num_parameters': 2177, 'accuracy': 0.629742033383915, 'precision': 0.7337278106508875, 'recall': 0.6169154228855721, 'f1_score': 0.6702702702702703, 'latency_ms_per_sample': 0.11014070062246598, 'auc': 0.7017732349923533}

Fold 2/5
Entrenando normal_model...
normal_model {'fold': 2, 'model': 'normal_model', 'num_features': 50, 'num_parameters': 54273, 'accuracy': 0.669195751138088, 'precision': 0.7288557213930348, 'recall': 0.7288557213930348, 'f1_score': 0.7288557213930348, 'latency_ms_per_sample': 0.11926934643110121, 'auc': 0.7388446870704841}
Entrenando tiny_model...

,fold,model,num_features,num_parameters,accuracy,precision,recall,f1_score,latency_ms_per_sample,auc
0,1,normal_model,50,54273,0.643399,0.753799,0.616915,0.678523,0.126300,0.713330
1,1,tiny_model,50,2177,0.629742,0.733728,0.616915,0.670270,0.110141,0.701773
2,2,normal_model,50,54273,0.669196,0.728856,0.728856,0.728856,0.119269,0.738845
3,2,tiny_model,50,2177,0.638847,0.757862,0.599502,0.669444,0.110164,0.691475
4,3,normal_model,50,54273,0.696510,0.770053,0.716418,0.742268,0.115623,0.763275
5,3,tiny_model,50,2177,0.672231,0.759777,0.676617,0.715789,0.121359,0.744236
6,4,normal_model,50,54273,0.673252,0.764205,0.670823,0.714475,0.121142,0.721834
7,4,tiny_model,50,2177,0.650456,0.756757,0.628429,0.686649,0.321881,0.711936
8,5,normal_model,50,54273,0.653495,0.772871,0.610973,0.682451,0.117658,0.721591
9,5,tiny_model,50,2177,0.656535,0.759644,0.638404,0.693767,0.106531,0.723929


Este último bloque tiene como objetivo sintetizar todos los resultados obtenidos durante la validación cruzada para poder comparar de forma clara y rigurosa los dos modelos: normal y tiny.

Hasta ahora, hemos generado múltiples resultados: en cada fold (5 en total) y para cada modelo, hemos calculado diferentes métricas. Esto significa que tenemos varias mediciones por modelo, no solo una. Para tomar una decisión informada, necesitamos resumir esa información.

Aquí es donde entra en juego groupby.

Primero, agrupamos los resultados por tipo de modelo:

results_df.groupby("model")

Esto separa automáticamente los resultados en dos grupos:

normal_model
tiny_model

A continuación, aplicamos funciones de agregación (agg) sobre cada métrica. En lugar de quedarnos con un único valor, calculamos:

la media (mean): indica el rendimiento promedio del modelo
la desviación estándar (std): indica la estabilidad del modelo entre folds

Esto es clave desde el punto de vista experimental. No solo queremos un modelo que funcione bien, sino que sea consistente.

Las métricas que se resumen son:

accuracy: proporción de aciertos globales
precision: calidad de las predicciones positivas
recall: capacidad de detectar positivos reales
f1_score: métrica principal (equilibrio entre precision y recall)
auc: capacidad de separación entre clases

Además, se incluyen dos aspectos fundamentales para escenarios edge:

num_parameters: tamaño del modelo (complejidad)
latency_ms_per_sample: tiempo medio de inferencia por muestra

En estos dos casos:

num_parameters se promedia (aunque suele ser constante por modelo)
latency se resume con media y desviación estándar

El resultado final (summary) es una tabla compacta donde cada fila corresponde a un modelo y cada columna describe su comportamiento medio y su variabilidad.

Este resumen permite hacer una comparación directa y fundamentada. Por ejemplo:

Si el modelo tiny tiene un F1-score ligeramente inferior pero una latencia mucho menor, puede ser preferible en un entorno real.
Si el modelo normal mejora significativamente el F1-score, quizá compense su mayor coste computacional.

En definitiva, este bloque convierte todos los experimentos en una visión clara que facilita la decisión final del proyecto:

elegir el modelo que mejor equilibra rendimiento predictivo y eficiencia computacional.

In [19]:
# ============================================================
# 6. Resumen de resultados
# ============================================================

summary = results_df.groupby("model").agg({
    "accuracy": ["mean", "std"],
    "precision": ["mean", "std"],
    "recall": ["mean", "std"],
    "f1_score": ["mean", "std"],
    "auc": ["mean", "std"],
    "num_parameters": "mean",
    "latency_ms_per_sample": ["mean", "std"]
})

summary

accuracy           precision              recall            \
                  mean       std      mean       std      mean       std   
model                                                                      
normal_model  0.667170  0.020335  0.757957  0.017832  0.668797  0.054576   
tiny_model    0.649562  0.016363  0.753553  0.011155  0.631974  0.028845   

              f1_score                 auc           num_parameters  \
                  mean       std      mean       std           mean   
model                                                                 
normal_model  0.709315  0.028126  0.731775  0.019906        54273.0   
tiny_model    0.687184  0.019122  0.714670  0.020443         2177.0   

             latency_ms_per_sample            
                              mean       std  
model                                         
normal_model              0.119998  0.004067  
tiny_model                0.154015  0.094005

Este bloque final cierra el pipeline de forma práctica: guarda todos los resultados y deja preparados los artefactos para su análisis posterior o para su uso en producción.

Hasta ahora hemos trabajado en memoria (variables como results_df y summary), pero en un proyecto real —y especialmente en un TFG o investigación— es fundamental persistir los resultados. Esto permite:

Reproducibilidad (no tener que reentrenar todo cada vez)
Análisis posterior (por ejemplo, en Excel o notebooks adicionales)
Documentación del experimento

Primero, definimos dos carpetas dentro del proyecto:

results: donde guardaremos métricas y tablas
models: pensada para guardar los modelos entrenados (aunque aquí aún no los exportamos explícitamente)

Esto se hace utilizando Path, igual que antes, manteniendo consistencia en la gestión de rutas.

RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = BASE_DIR / "models"

A continuación, usamos .mkdir(exist_ok=True) para crear estas carpetas si no existen. Este detalle es importante: evita errores si el código se ejecuta varias veces.

Después, guardamos los resultados en formato CSV:

results_df: contiene todas las métricas de cada fold y cada modelo (nivel detallado)
summary: contiene el resumen agregado (nivel global)

Esto se traduce en dos archivos:

kfold_results_happy_vs_not_happy.csv
summary_happy_vs_not_happy.csv


Finalmente, se imprime la ruta donde se han guardado los resultados, lo que facilita al usuario localizarlos rápidamente.

En términos de workflow, este bloque representa el cierre del experimento:

Ya hemos construido los datos
Hemos entrenado y evaluado los modelos
Hemos comparado su rendimiento
Y ahora dejamos todo registrado

El siguiente paso natural (fuera de este bloque) sería:

Analizar los CSV para tomar una decisión final
Exportar el mejor modelo (por ejemplo, el tiny si es competitivo)
Prepararlo para despliegue en entorno edge

Este tipo de organización es exactamente lo que se espera en un proyecto bien estructurado: no solo obtener resultados, sino gestionarlos de forma clara, reproducible y profesional.

In [20]:
# ============================================================
# 7. Guardar resultados
# ============================================================

from pathlib import Path

RESULTS_DIR = BASE_DIR / "results"
MODELS_DIR = BASE_DIR / "models"

RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

results_df.to_csv(RESULTS_DIR / "kfold_results_happy_vs_not_happy.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary_happy_vs_not_happy.csv")

print("Resultados guardados en:", RESULTS_DIR)

Resultados guardados en: C:\Users\pelli\Documents\TFG_Emociones\results


Este bloque tiene un objetivo muy claro dentro del workflow: guardar el mejor modelo tiny que hemos encontrado durante todo el proceso de validación cruzada.

Hasta ahora, no nos hemos quedado con un único modelo, sino que hemos entrenado varios (uno por cada fold). Sin embargo, durante el entrenamiento hemos ido monitorizando cuál es el mejor modelo tiny en función de la métrica más importante: el F1-score.

En el bloque anterior ya se fue actualizando esta información:

best_tiny_model: guarda el modelo con mejor rendimiento
best_tiny_f1: guarda su F1-score
best_fold: indica en qué fold se obtuvo

Ahora lo que hacemos es persistir ese modelo en disco, para poder reutilizarlo sin necesidad de reentrenar.

Primero, construimos la ruta donde se guardará el modelo:

best_model_path = MODELS_DIR / f"best_tiny_model_fold_{best_fold}.keras"

Aquí hay dos detalles importantes:

Se guarda dentro de la carpeta models, manteniendo el proyecto organizado
El nombre incluye el número de fold, lo que aporta trazabilidad (sabemos de dónde viene ese modelo)

Después, utilizamos:

best_tiny_model.save(best_model_path)

Esto guarda el modelo en formato .keras, que es el formato nativo de TensorFlow / Keras. Este formato incluye:

La arquitectura del modelo
Los pesos entrenados
La configuración del entrenamiento

Es decir, permite cargar el modelo más adelante directamente con:

tf.keras.models.load_model(...)

sin necesidad de reconstruir nada.

Finalmente, se imprimen tres valores:

El fold donde se obtuvo el mejor modelo
El mejor F1-score alcanzado
La ruta donde se ha guardado el modelo

Esto no es solo informativo, también es útil para documentar resultados.

Desde el punto de vista del objetivo del proyecto, este bloque es fundamental:

Ya no solo has comparado modelos, sino que has seleccionado y guardado el mejor candidato para despliegue en entorno edge.

Esto conecta directamente con la idea central del experimento:

comprobar si un modelo tiny puede ofrecer un rendimiento competitivo con menor coste computacional.

A partir de aquí, los siguientes pasos naturales serían:

Medir su rendimiento en un entorno real
Convertirlo (por ejemplo a TensorFlow Lite)
Integrarlo en un sistema edge (móvil, embebido, etc.)

Este es el punto donde tu pipeline pasa de ser experimental a ser aplicable en un caso real.

In [21]:
# ============================================================
# 8. Guardar mejor modelo tiny
# ============================================================

best_model_path = MODELS_DIR / f"best_tiny_model_fold_{best_fold}.keras"
best_tiny_model.save(best_model_path)

print("Mejor fold tiny:", best_fold)
print("Mejor F1 tiny:", best_tiny_f1)
print("Modelo guardado en:", best_model_path)

Mejor fold tiny: 3
Mejor F1 tiny: 0.7157894736842105
Modelo guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model_fold_3.keras


Este bloque representa el paso final para llevar tu modelo desde un entorno de investigación a un entorno real: la conversión a TensorFlow Lite, que es el formato optimizado para dispositivos edge.

Hasta ahora, hemos trabajado con modelos en formato estándar de TensorFlow / Keras, ideales para entrenamiento y evaluación. Sin embargo, estos modelos no están optimizados para ejecutarse en dispositivos con recursos limitados como:

móviles
sistemas embebidos
dispositivos IoT

Aquí es donde entra TensorFlow Lite (TFLite).

¿Qué hace este bloque?

Primero, se crea un converter a partir del modelo Keras:

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)

Este objeto es el encargado de transformar el modelo a un formato más ligero y eficiente.

A continuación, se realiza la conversión:

tflite_model = converter.convert()

Este paso genera una versión optimizada del modelo que:

ocupa menos espacio en memoria
tiene menor latencia de inferencia
está diseñada para ejecutarse sin necesidad de todo el ecosistema de TensorFlow
Guardado del modelo

Después, se define la ruta donde se guardará:

tflite_path = MODELS_DIR / "best_tiny_model.tflite"

Y se escribe el archivo en disco:

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

El resultado es un archivo .tflite, que es el formato estándar para despliegue en edge.

¿Por qué es tan importante este paso?

Este bloque conecta directamente con el objetivo principal del proyecto:

evaluar si un modelo tiny puede ser no solo preciso, sino también desplegable en entornos reales.

Con este archivo .tflite, ya puedes:

Integrarlo en una app móvil (Android, por ejemplo)
Ejecutarlo en dispositivos como Raspberry Pi
Usarlo en sistemas en tiempo real con restricciones de latencia


Hasta aquí, el workflow completo ha sido:

construir datos
entrenar modelos
evaluarlos rigurosamente
seleccionar el mejor
y finalmente adaptarlo para el mundo real


Este bloque convierte tu modelo en algo que puede salir del notebook y funcionar en producción.

In [22]:
# ============================================================
# 9. Exportar mejor modelo tiny a TensorFlow Lite
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)
tflite_model = converter.convert()

tflite_path = MODELS_DIR / "best_tiny_model.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Modelo TFLite guardado en:", tflite_path)

INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmpb1wvf3xi\assets


INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmpb1wvf3xi\assets


Saved artifact at 'C:\Users\pelli\AppData\Local\Temp\tmpb1wvf3xi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50), dtype=tf.float32, name='keras_tensor_29')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1939912627728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912626768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912628496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912637712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912635216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912638288: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo TFLite guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model.tflite


Este bloque (opcional) introduce una optimización muy importante cuando pensamos en despliegue real: la cuantización del modelo.

Hasta ahora ya teníamos una versión en TensorFlow Lite, pero aquí damos un paso más: reducir aún más el tamaño y mejorar la eficiencia, sacrificando (ligeramente) la precisión si es necesario.

¿Qué es la cuantización?

La cuantización consiste en representar los pesos del modelo con menor precisión numérica. En lugar de usar números en coma flotante de 32 bits (float32), el modelo puede usar representaciones más compactas (por ejemplo, 8 bits).

Esto tiene varias ventajas:

Reduce el tamaño del modelo
Disminuye el uso de memoria
Acelera la inferencia en hardware limitado
Mejora la eficiencia energética
¿Qué hace exactamente este código?

Primero, igual que antes, se crea el converter:

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)

La diferencia clave está aquí:

converter.optimizations = [tf.lite.Optimize.DEFAULT]

Esta línea activa las optimizaciones automáticas de TensorFlow Lite, que incluyen cuantización por defecto (post-training quantization).

Después, se realiza la conversión:

tflite_quant_model = converter.convert()

Esto genera una versión optimizada del modelo, normalmente más pequeña y rápida que la versión TFLite estándar.

Guardado del modelo cuantizado

Se guarda en un archivo diferente:

tflite_quant_path = MODELS_DIR / "best_tiny_model_quantized.tflite"

Esto es importante porque ahora tienes dos versiones del modelo:

best_tiny_model.tflite → modelo estándar
best_tiny_model_quantized.tflite → modelo optimizado
¿Por qué mantener ambas versiones?

Porque en un escenario real hay que comparar:

¿Cuánto se reduce el tamaño?
¿Cuánto mejora la latencia?
¿Cuánto empeora (si lo hace) el F1-score?

Esto encaja perfectamente con el enfoque experimental del proyecto: no solo optimizar, sino medir el impacto de cada decisión.



Este bloque representa el último nivel de optimización del pipeline:

Primero entrenamos modelos (normal vs tiny)
Luego elegimos el mejor tiny
Lo convertimos a TFLite
Y ahora lo optimizamos para edge

El mensaje importante es:

en sistemas reales, no basta con que un modelo funcione bien; tiene que hacerlo rápido, ligero y eficiente.

A partir de aquí, ya tienes todo lo necesario para:

desplegar el modelo en un dispositivo real
comparar versiones (normal vs tiny vs cuantizado)
tomar una decisión final basada en rendimiento y eficiencia

In [23]:
# ============================================================
# 10. Exportar versión cuantizada opcional
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(best_tiny_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

tflite_quant_path = MODELS_DIR / "best_tiny_model_quantized.tflite"

with open(tflite_quant_path, "wb") as f:
    f.write(tflite_quant_model)

print("Modelo TFLite cuantizado guardado en:", tflite_quant_path)

INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmpjk4q7nwf\assets


INFO:tensorflow:Assets written to: C:\Users\pelli\AppData\Local\Temp\tmpjk4q7nwf\assets


Saved artifact at 'C:\Users\pelli\AppData\Local\Temp\tmpjk4q7nwf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50), dtype=tf.float32, name='keras_tensor_29')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1939912627728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912626768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912628496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912637712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912635216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1939912638288: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo TFLite cuantizado guardado en: C:\Users\pelli\Documents\TFG_Emociones\models\best_tiny_model_quantized.tflite


Si eres capaz de entender este pipeline de principio a fin —desde la carga del dataset, la construcción de características, el diseño experimental con k-fold, el entrenamiento de modelos, la evaluación rigurosa y finalmente la exportación a edge— ya estás trabajando a un nivel muy alto.

Pero hay algo aún más importante en el contexto de un TFG:

no se trata solo de que funcione, sino de que seas capaz de explicarlo, justificarlo y documentarlo correctamente.

Si consigues:

Explicar el flujo completo de datos (de vídeo → features → modelo)
Justificar las decisiones metodológicas (k-fold, selección de características, métricas)
Comparar modelos (normal vs tiny) de forma rigurosa
Integrar el modelo tiny en una aplicación real (aunque sea un prototipo sencillo)
Documentar todo esto de forma clara en la memoria

entonces ya tienes un TFG de calidad muy elevada.

¿Qué podrías hacer como mejora?

Una vez tengas todo lo anterior bien cerrado y documentado, entonces sí tiene sentido plantear extensiones. Algunas ideas naturales serían:

1. Comparar modelo tiny vs versión cuantizada

Ya has generado ambos modelos (.tflite y .tflite cuantizado). El siguiente paso sería evaluarlos en igualdad de condiciones:

Ejecutar ambos con el intérprete de TensorFlow Lite
Medir:
F1-score (mantener rendimiento)
Latencia por muestra (tiempo de inferencia)
Tamaño del modelo

El objetivo es responder a una pregunta muy clara:

¿La cuantización mejora la eficiencia sin degradar significativamente el rendimiento?

Esto refuerza mucho el valor aplicado del trabajo.

2. Añadir una nueva modalidad (por ejemplo, acústica)

El dataset que estás utilizando no es solo visual. También incluye información acústica (audio).

Podrías extender el pipeline de esta forma:

Cargar el archivo CMU_MOSEI_COVAREP.csd
Aplicar el mismo enfoque:
agregación temporal (mean, std)
Generar nuevas features acústicas
Combinar con las visuales:
features_final = [features_visuales | features_acústicas]

Y volver a entrenar el modelo.

Esto te permitiría explorar:

¿Mejora la detección de emociones al usar múltiples modalidades?

Este tipo de extensión tiene mucho valor académico porque conecta con el concepto de aprendizaje multimodal.



Mensaje final

Todas estas mejoras son interesantes, pero hay una prioridad clara:

Primero cierra bien el pipeline actual y documéntalo correctamente.

Un TFG excelente no es el que tiene más cosas, sino el que:

está bien planteado
es metodológicamente sólido
y se explica con claridad

Si haces eso, todo lo demás suma. Si no, incluso una implementación más compleja pierde valor.



En resumen:

Ya tienes un pipeline completo y profesional
Ya puedes entrenar, evaluar y desplegar modelos
Ya puedes tomar decisiones basadas en métricas

A partir de aquí, estás pasando de “hacer un proyecto” a hacer ingeniería y ciencia aplicada.